# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MTahaR-dev/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

**Provisional choice:** Lane 2 — Refresh / Content Opportunity Scoring.

This notebook frames my Week 1 question and checks three supporting counts from the starter data. It does not train a model. I used an AI assistant to help draft the framing and run the summaries, after reading `skills/README.md`, `framing-ml-problems`, and `flyrank/flyrank-data`.

**Rerun:** Run all cells in order with Python 3 and pandas. A local clone uses the bundled CSV; Colab can load the same CSV from a pinned repository revision. References: `docs/ml-intern-dataset-and-lane-guide.md` and `docs/data-dictionary.md`.

## 1. My lane (or freestyle) and why

My provisional lane is **Lane 2: Refresh / Content Opportunity Scoring**. I want to help a content editor decide which pages to review first when review time is limited. The starter dataset contains search exposure, recent impression changes, and time since the last update, so I can investigate how these measured signals help prioritize a review queue. My initial focus is pages that still receive search impressions but show an observed downward impression trend, with update age providing additional review context. The quick look in Section 3 will check whether this is a useful starting point. Over the next seven weeks, I plan to start with a transparent rule and test whether more detailed analysis or ML improves prioritization. I am choosing a decision-support problem: a high priority score would prompt a human review, and would not guarantee that editing the page will improve performance. This choice is provisional and can change through the end of Week 4.

## 2. The question: decision, action, cost of a wrong call

**Research question:** Which pages with ongoing search exposure and a recent decline in impressions should a content editor review first for a possible refresh?

- **Who and decision:** A content editor or SEO analyst deciding how to allocate limited page-review time.
- **Unit of analysis:** One pseudonymized content item (page) at a review decision point. The starter data has one row per item, with trailing-90-day totals and two consecutive 30-day comparison windows.
- **Output:** A ranked page-review queue with a priority score, measured reason codes, and uncertainty notes. The initial action is review for a possible refresh.
- **Action:** The editor inspects the page and its context, then decides whether to update it, investigate further, or monitor it. The queue does not automatically edit or remove content.
- **Cost of a wrong call:** A false positive wastes editorial time and may encourage unnecessary changes to healthy content. A false negative leaves a page that warrants attention unreviewed, potentially allowing losses to continue. I cannot estimate hours, revenue, or refresh return from this dataset.

**Why data or ML can help:** Measured exposure, impression changes, and update age allow a consistent comparison across many pages. I will first build a transparent rule, because it may already be sufficient. ML earns a place only if combining additional signals improves the queue against that baseline under a defensible evaluation. Choosing a model or reproducing a trend flag is not the research goal.

**Proposed success measure:** Precision@20 would match an assumed capacity of 20 reviews per cycle; this is a provisional capacity assumption, not a known business constraint. It needs an independent definition of a relevant candidate. For later outcome-based evaluation, I plan to define sustained future impression decline from warehouse daily data, using non-overlapping feature and outcome windows and minimum exposure. That would measure future-risk prioritization, not the benefit of editing. The starter snapshot cannot compute that future-outcome metric today. I will settle the relevance definition, windows, and baseline before model training, check held-out clients/time periods, and inspect the top 20 recommendations.

## 3. Quick look at the data (2-3 real numbers)

The following cells load the raw starter CSV and produce three counts. For the initial screen, “ongoing exposure” means at least **300 impressions in 90 days**. “Not recently updated” means at least **90 days since the last update**. These are provisional review policies, not evidence that a page is outdated.

`trend_direction == "down"` describes a drop of more than 20% in impressions in the most recent 30 days compared with the preceding 30 days. It is an observed trend bucket, not an independently measured future outcome.

In [1]:
from pathlib import Path
import pandas as pd

# Locate the existing starter CSV from the repo root or a notebook folder.
relative_path = Path("data/raw/content_refresh_anonymized.csv")
data_path = next(
    (parent / relative_path for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / relative_path).is_file()),
    None,
)

# Colab fallback: use the same fixed starter-data revision, without credentials.
starter_revision = "224d6365f72f39b6ce899376f8fe7b6d3a34265e"
starter_url = (
    "https://raw.githubusercontent.com/MTahaR-dev/flyrank-ml-internship/"
    f"{starter_revision}/{relative_path.as_posix()}"
)
df = pd.read_csv(data_path if data_path is not None else starter_url)

# Verify the page grain before counting; do not print individual records.
assert df["content_id"].notna().all(), "Missing page IDs need investigation."
assert df["content_id"].is_unique, "Duplicate page IDs need investigation."
print("Loaded the starter CSV; verified one row per content item.")

Loaded the starter CSV; verified one row per content item.


In [2]:
MIN_IMPRESSIONS_90D = 300
MIN_DAYS_SINCE_UPDATE = 90

observed_down = df["trend_direction"].eq("down")
review_screen = (
    observed_down
    & df["impressions_90d"].ge(MIN_IMPRESSIONS_90D)
    & df["days_since_last_update"].ge(MIN_DAYS_SINCE_UPDATE)
)

page_count = len(df)
down_count = int(observed_down.sum())
screen_count = int(review_screen.sum())

print(f"1. Starter inventory: {page_count:,} pages.")
print(f"2. Observed down trend: {down_count:,} pages "
      f"({down_count / page_count:.2%} of the starter inventory).")
print(f"3. Down trend + at least {MIN_IMPRESSIONS_90D} impressions/90d "
      f"+ at least {MIN_DAYS_SINCE_UPDATE} days since update: "
      f"{screen_count:,} pages ({screen_count / page_count:.2%} of the inventory).")

1. Starter inventory: 30,000 pages.
2. Observed down trend: 16,262 pages (54.21% of the starter inventory).
3. Down trend + at least 300 impressions/90d + at least 90 days since update: 4,467 pages (14.89% of the inventory).


**What these numbers suggest:** The inventory is large enough for prioritization to matter. The down-trend count shows that a broad trend flag alone leaves many pages to review. Even after adding exposure and update-age filters, thousands of candidates remain. This makes a ranked, explainable review queue worth investigating over the next seven weeks.

The screened pages are **review candidates**, not confirmed refresh opportunities. The counts motivate the question; they do not show that ML beats a rule, identify a cause of decline, or estimate recoverable traffic. The starter slice contains only pages with some impressions and content age of at least 90 days, so it is not representative of the entire content inventory.

## 4. Careful words: what I can and can't claim

**I can claim:** In this starter slice, I measured the counts printed above. The observed combination of exposure, impression decline, and update age suggests a useful starting screen for human review. Later results can support directional associations and decision-support recommendations within the data and validation scope actually tested.

**I cannot claim:** A page is outdated simply because it has not been updated recently; a refresh will recover traffic; editing caused recovery; these signals reveal Google's algorithm; or starter results automatically generalize to the full warehouse, unseen clients, or future periods. I have not trained or validated a model here.

**Important checks for later work:**

- Seasonality, low-volume noise, or related pages absorbing traffic may explain a decline. The starter snapshot alone cannot rule those explanations out.
- The starter `is_declining_label` is derived from `trend_direction`, which is derived from `trend_pct`. Those trend fields can describe today's screen, but must not be features in a model predicting that label. That label is not evidence of future decline or refresh effectiveness.
- For future-outcome modeling, all features must be available before the decision point and exclude the outcome window. Whole-client and time-based checks will help test generalization.
- IDs are only for grouping and joining. Missing measurements are not automatically zeros; `avg_position = 0` means no position data. Rate columns such as CTR use percentage units.
- Outputs will use aggregate counts and pseudonymized IDs only, with human review required before content changes.

## 5. Self-check

- [x] Selected one predefined lane and explained the provisional choice.
- [x] Named the question, decision, person, action, unit, output, and cost of errors.
- [x] Explained why a transparent rule comes first and why ML must earn its place.
- [x] Loaded the starter CSV and printed three supporting counts.
- [x] Executed every code cell in order with no errors and retained the outputs.
- [x] Outputs contain aggregate measurements, with no client names, page URLs, or private queries from the data.
- [x] Used observed, directional, and decision-support language; did not claim causation.

**Submission step:** Commit and push this executed notebook at `work/notebooks/w01_research_question.ipynb`, verify its outputs on GitHub, then submit the repository URL on the assignment card. This step is pending until the commit and push are completed.